# Exploratory analysis: dynamic organization

This notebook is outside the main paper workflow. Existing results and metric variants are historical/exploratory, not the authoritative paper results. Original analysis cells and outputs are retained. Some legacy sections require selective execution; this is not a verified clean-run pipeline.

The main workflow is in `notebooks/paper/`. This notebook may write legacy exports under `analysis_exports/`; the paper pipeline uses its own `outputs/paper/` directory.

Plot defaults now come from `plot_style.py`. Prior static image outputs were cleared; rerun plot cells after their prerequisites to see the shared style. Specialized heatmap scales and animations retain their own semantic encodings.

In [ ]:
from pathlib import Path
import os

PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "data/wired").is_dir() and (p / "paper").is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError("Run from this repository or a notebook directory inside it.")
os.chdir(PROJECT_ROOT)  # Preserve project-relative paths when launched from a subdirectory.
print("Project root:", PROJECT_ROOT)

# Shared project plotting conventions.
import sys
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
from plot_style import (
    apply_style, expertise_palette, EXPERTISE_COLORS, LEVEL_LABELS,
    METRIC_LABELS, MODEL_COLORS, MODEL_MARKERS, PRIMARY, NEUTRAL,
    plot_metric_trajectories, save_figure,
)
apply_style()


In [1]:
from pathlib import Path

import json
import numpy as np
import pandas as pd

from scipy.spatial.distance import (
    pdist,
    squareform,
)

EXPORT_DIR = Path(
    "analysis_exports"
)

In [2]:
turns_dynamic = pd.read_csv(
    EXPORT_DIR / "turns_dynamic.csv"
)

X_turns = np.load(
    EXPORT_DIR / "turn_embeddings.npy"
)

with open(
    EXPORT_DIR / "dynamic_manifest.json",
    "r",
    encoding="utf-8",
) as file:
    manifest = json.load(file)

manifest

{'n_turns': 2078,
 'embedding_rows': 2078,
 'embedding_dimensions': 384,
 'distance': 'euclidean_chord_on_unit_embeddings',
 'turn_unit': 'collapsed consecutive same-speaker utterances',
 'roles': ['expert', 'partner']}

In [3]:
assert (
    turns_dynamic[
        "_embedding_row"
    ].max()
    < len(X_turns)
)

assert len(turns_dynamic) == manifest[
    "n_turns"
]

turns_dynamic[
    "_embedding_row"
] = turns_dynamic[
    "_embedding_row"
].astype(int)

turns_dynamic[
    "_turn_order"
] = turns_dynamic[
    "_turn_order"
].astype(int)

In [4]:
norms = np.linalg.norm(
    X_turns,
    axis=1,
    keepdims=True,
)

if np.any(norms < 1e-12):
    raise ValueError(
        "At least one turn embedding has zero norm."
    )

X_turns = X_turns / norms

In [5]:
print(
    "Turns:",
    len(turns_dynamic),
)

print(
    "Conversations:",
    turns_dynamic[
        "conversation_uid"
    ].nunique(),
)

print(
    "Videos/topics:",
    turns_dynamic[
        "video_uid"
    ].nunique(),
)

print(
    turns_dynamic[
        "level_label"
    ].value_counts()
)

print(
    turns_dynamic[
        "_role"
    ].value_counts()
)

Turns: 2078
Conversations: 80
Videos/topics: 16
level_label
child            479
graduate         413
expert           400
undergraduate    394
teenager         392
Name: count, dtype: int64
_role
expert     1056
partner    1022
Name: count, dtype: int64


In [6]:
#recurrence determinism helper
EPSILON = 1e-12


def recurrent_run_points(
    boolean_sequence,
    minimum_length=2,
):
    """
    Number of True values belonging to runs with at least
    minimum_length consecutive True values.
    """

    total = 0
    current_run = 0

    for value in boolean_sequence:
        if value:
            current_run += 1
        else:
            if current_run >= minimum_length:
                total += current_run

            current_run = 0

    if current_run >= minimum_length:
        total += current_run

    return total

In [7]:
def recurrence_determinism(
    distance_matrix,
    recurrence_threshold,
    theiler_window=1,
    minimum_line_length=2,
):
    """
    Proportion of recurrent points belonging to diagonal
    recurrence sequences.

    Theiler window excludes the main diagonal and immediate
    temporal neighbors.
    """

    n = len(distance_matrix)

    total_recurrent = 0
    deterministic_recurrent = 0

    for offset in range(
        theiler_window + 1,
        n,
    ):
        diagonal = np.diag(
            distance_matrix,
            k=offset,
        )

        recurrent = (
            diagonal
            <= recurrence_threshold
        )

        total_recurrent += recurrent.sum()

        deterministic_recurrent += (
            recurrent_run_points(
                recurrent,
                minimum_length=minimum_line_length,
            )
        )

    if total_recurrent == 0:
        return np.nan

    return (
        deterministic_recurrent
        / total_recurrent
    )

calculate organization from a distance matrix


In [8]:
def sequence_organization_metrics(
    distance_matrix,
    maximum_lag,
    recurrence_threshold,
):
    n = len(distance_matrix)

    upper_triangle = distance_matrix[
        np.triu_indices(
            n,
            k=1,
        )
    ]

    mean_pairwise_distance = (
        upper_triangle.mean()
    )

    adjacent_distances = np.diag(
        distance_matrix,
        k=1,
    )

    mean_adjacent_distance = (
        adjacent_distances.mean()
    )

    if mean_pairwise_distance > EPSILON:
        temporal_locality = (
            1
            - mean_adjacent_distance
            / mean_pairwise_distance
        )
    else:
        temporal_locality = np.nan

    lags = np.arange(
        1,
        maximum_lag + 1,
    )

    lag_distances = np.array([
        np.diag(
            distance_matrix,
            k=lag,
        ).mean()
        for lag in lags
    ])

    if mean_pairwise_distance > EPSILON:
        normalized_lag_distances = (
            lag_distances
            / mean_pairwise_distance
        )
    else:
        normalized_lag_distances = (
            np.full_like(
                lag_distances,
                np.nan,
            )
        )

    valid = np.isfinite(
        normalized_lag_distances
    )

    if valid.sum() >= 2:
        lag_growth_slope = np.polyfit(
            np.log1p(lags[valid]),
            normalized_lag_distances[
                valid
            ],
            deg=1,
        )[0]

        lag_contrast = (
            normalized_lag_distances[
                valid
            ][-1]
            - normalized_lag_distances[
                valid
            ][0]
        )
    else:
        lag_growth_slope = np.nan
        lag_contrast = np.nan

    determinism = recurrence_determinism(
        distance_matrix,
        recurrence_threshold,
    )

    metrics = {
        "mean_pairwise_distance": (
            mean_pairwise_distance
        ),
        "mean_adjacent_distance": (
            mean_adjacent_distance
        ),
        "temporal_locality": (
            temporal_locality
        ),
        "lag_growth_slope": (
            lag_growth_slope
        ),
        "lag_contrast": (
            lag_contrast
        ),
        "recurrence_determinism": (
            determinism
        ),
    }

    return (
        metrics,
        normalized_lag_distances,
    )

In [9]:
def role_preserving_permutation(
    roles,
    random_generator,
):
    """
    Keep expert/partner position pattern fixed while
    shuffling embeddings within each role.
    """

    roles = np.asarray(roles)

    permutation = np.arange(
        len(roles)
    )

    for role in np.unique(roles):
        positions = np.flatnonzero(
            roles == role
        )

        permutation[positions] = (
            random_generator.permutation(
                positions
            )
        )

    return permutation

In [10]:
ORGANIZATION_METRICS = [
    "temporal_locality",
    "lag_growth_slope",
    "lag_contrast",
    "recurrence_determinism",
]

In [11]:
def analyze_organization_sequence(
    points,
    roles,
    random_generator,
    n_shuffles=500,
    maximum_lag=6,
    recurrence_rate=0.10,
):
    points = np.asarray(
        points,
        dtype=float,
    )

    roles = np.asarray(roles)

    distance_matrix = squareform(
        pdist(
            points,
            metric="euclidean",
        )
    )

    all_pairwise_distances = (
        distance_matrix[
            np.triu_indices(
                len(points),
                k=1,
            )
        ]
    )

    # Same threshold is retained for observed and shuffled
    # sequences because the point cloud does not change.
    recurrence_threshold = np.quantile(
        all_pairwise_distances,
        recurrence_rate,
    )

    observed, observed_lags = (
        sequence_organization_metrics(
            distance_matrix,
            maximum_lag=maximum_lag,
            recurrence_threshold=(
                recurrence_threshold
            ),
        )
    )

    null_metrics = {
        metric: np.full(
            n_shuffles,
            np.nan,
        )
        for metric in ORGANIZATION_METRICS
    }

    null_lags = np.full(
        (
            n_shuffles,
            maximum_lag,
        ),
        np.nan,
    )

    for shuffle_index in range(
        n_shuffles
    ):
        permutation = (
            role_preserving_permutation(
                roles,
                random_generator,
            )
        )

        shuffled_distance_matrix = (
            distance_matrix[
                np.ix_(
                    permutation,
                    permutation,
                )
            ]
        )

        shuffled_metrics, shuffled_lags = (
            sequence_organization_metrics(
                shuffled_distance_matrix,
                maximum_lag=maximum_lag,
                recurrence_threshold=(
                    recurrence_threshold
                ),
            )
        )

        for metric in ORGANIZATION_METRICS:
            null_metrics[metric][
                shuffle_index
            ] = shuffled_metrics[metric]

        null_lags[
            shuffle_index
        ] = shuffled_lags

    summary = {
        "n_turns": len(points),
        "maximum_lag": maximum_lag,
        "recurrence_threshold": (
            recurrence_threshold
        ),
        "mean_pairwise_distance": observed[
            "mean_pairwise_distance"
        ],
        "mean_adjacent_distance": observed[
            "mean_adjacent_distance"
        ],
    }

    for metric in ORGANIZATION_METRICS:
        observed_value = observed[
            metric
        ]

        null_values = null_metrics[
            metric
        ]

        null_values = null_values[
            np.isfinite(null_values)
        ]

        if (
            np.isfinite(observed_value)
            and len(null_values) >= 2
        ):
            null_mean = null_values.mean()
            null_sd = null_values.std(
                ddof=1
            )

            if null_sd > EPSILON:
                z_score = (
                    observed_value
                    - null_mean
                ) / null_sd
            else:
                z_score = np.nan

            # One-sided permutation p-value:
            # observed organization greater than shuffled
            p_value = (
                1
                + np.sum(
                    null_values
                    >= observed_value
                )
            ) / (
                len(null_values) + 1
            )
        else:
            null_mean = np.nan
            null_sd = np.nan
            z_score = np.nan
            p_value = np.nan

        summary[
            f"{metric}_observed"
        ] = observed_value

        summary[
            f"{metric}_null_mean"
        ] = null_mean

        summary[
            f"{metric}_null_sd"
        ] = null_sd

        summary[
            f"{metric}_z"
        ] = z_score

        summary[
            f"{metric}_p_perm"
        ] = p_value

    lag_summary = []

    for lag_index in range(
        maximum_lag
    ):
        lag = lag_index + 1

        null_values = null_lags[
            :,
            lag_index,
        ]

        null_values = null_values[
            np.isfinite(null_values)
        ]

        observed_value = observed_lags[
            lag_index
        ]

        null_mean = (
            null_values.mean()
            if len(null_values) > 0
            else np.nan
        )

        null_sd = (
            null_values.std(ddof=1)
            if len(null_values) > 1
            else np.nan
        )

        if (
            np.isfinite(observed_value)
            and np.isfinite(null_sd)
            and null_sd > EPSILON
        ):
            lag_z = (
                observed_value
                - null_mean
            ) / null_sd
        else:
            lag_z = np.nan

        lag_summary.append({
            "lag": lag,
            "lag_distance_observed": (
                observed_value
            ),
            "lag_distance_null_mean": (
                null_mean
            ),
            "lag_distance_null_sd": (
                null_sd
            ),
            "lag_distance_z": lag_z,
        })

    return summary, lag_summary

In [12]:
trajectory_roles = {
    "joint": None,
    "expert": "expert",
    "partner": "partner",
}

In [13]:
def calculate_organization_table(
    turn_table,
    embeddings,
    group_columns,
    n_shuffles=500,
    maximum_lag=6,
    minimum_turns=10,
    recurrence_rate=0.10,
    random_seed=42,
):
    random_generator = (
        np.random.default_rng(
            random_seed
        )
    )

    result_rows = []
    lag_rows = []

    grouped = turn_table.groupby(
        group_columns,
        sort=True,
        dropna=False,
    )

    for group_key, group in grouped:
        if not isinstance(
            group_key,
            tuple,
        ):
            group_key = (
                group_key,
            )

        metadata = dict(
            zip(
                group_columns,
                group_key,
            )
        )

        group = (
            group
            .sort_values("_turn_order")
            .reset_index(drop=True)
        )

        for trajectory_type, role in (
            trajectory_roles.items()
        ):
            if role is None:
                trajectory = group.copy()
            else:
                trajectory = (
                    group[
                        group["_role"] == role
                    ]
                    .copy()
                    .reset_index(drop=True)
                )

            if len(trajectory) < minimum_turns:
                continue

            embedding_rows = (
                trajectory[
                    "_embedding_row"
                ]
                .to_numpy(dtype=int)
            )

            points = embeddings[
                embedding_rows
            ]

            roles = trajectory[
                "_role"
            ].to_numpy()

            summary, lag_summary = (
                analyze_organization_sequence(
                    points=points,
                    roles=roles,
                    random_generator=(
                        random_generator
                    ),
                    n_shuffles=n_shuffles,
                    maximum_lag=maximum_lag,
                    recurrence_rate=(
                        recurrence_rate
                    ),
                )
            )

            result_row = {
                **metadata,
                "trajectory_type": (
                    trajectory_type
                ),
                **summary,
            }

            result_rows.append(
                result_row
            )

            for lag_result in lag_summary:
                lag_rows.append({
                    **metadata,
                    "trajectory_type": (
                        trajectory_type
                    ),
                    "n_turns": len(
                        trajectory
                    ),
                    **lag_result,
                })

    return (
        pd.DataFrame(result_rows),
        pd.DataFrame(lag_rows),
    )

In [14]:
conversation_group_columns = [
    "video_uid",
    "conversation_uid",
    "conversation_id",
    "level_label",
]

organization_conversation_test, lag_curves_test = (
    calculate_organization_table(
        turn_table=turns_dynamic,
        embeddings=X_turns,
        group_columns=(
            conversation_group_columns
        ),
        n_shuffles=50,
        maximum_lag=6,
        minimum_turns=10,
        random_seed=42,
    )
)

organization_conversation_test.head()

,video_uid,conversation_uid,conversation_id,level_label,trajectory_type,n_turns,maximum_lag,recurrence_threshold,mean_pairwise_distance,mean_adjacent_distance,...,lag_contrast_observed,lag_contrast_null_mean,lag_contrast_null_sd,lag_contrast_z,lag_contrast_p_perm,recurrence_determinism_observed,recurrence_determinism_null_mean,recurrence_determinism_null_sd,recurrence_determinism_z,recurrence_determinism_p_perm
0,astro,astro::wired_astro_12,wired_astro_12,child,joint,23,6,0.980034,1.203716,1.085090,...,0.098857,-0.000212,0.024925,3.974697,0.019608,0.181818,0.087034,0.087489,1.083385,0.117647
1,astro,astro::wired_astro_12,wired_astro_12,child,expert,11,6,0.897923,1.101134,1.006016,...,0.130337,0.006515,0.066474,1.862705,0.058824,0.000000,0.167333,0.234302,-0.714177,1.000000
2,astro,astro::wired_astro_12,wired_astro_12,child,partner,12,6,1.132220,1.291462,1.276323,...,0.024041,0.005330,0.035031,0.534130,0.313725,0.333333,0.046000,0.127578,2.252214,0.137255
3,astro,astro::wired_astro_13,wired_astro_13,teenager,joint,13,6,0.966636,1.176068,1.165648,...,0.029008,-0.033242,0.039581,1.572743,0.098039,0.000000,0.000000,0.000000,NaN,1.000000
4,astro,astro::wired_astro_14,wired_astro_14,undergraduate,joint,17,6,0.845830,1.142912,0.979015,...,0.146167,0.022423,0.038812,3.188255,0.019608,0.000000,0.152358,0.145578,-1.046578,1.000000


In [15]:
organization_conversation, lag_curves_conversation = (
    calculate_organization_table(
        turn_table=turns_dynamic,
        embeddings=X_turns,
        group_columns=(
            conversation_group_columns
        ),
        n_shuffles=500,
        maximum_lag=6,
        minimum_turns=10,
        random_seed=42,
    )
)

In [16]:
organization_conversation.to_csv(
    EXPORT_DIR
    / "organization_conversation.csv",
    index=False,
)

lag_curves_conversation.to_csv(
    EXPORT_DIR
    / "organization_lag_curves.csv",
    index=False,
)

In [17]:
organization_coverage = (
    organization_conversation
    .groupby(
        "trajectory_type"
    )
    .agg(
        rows=(
            "conversation_uid",
            "size",
        ),
        conversations=(
            "conversation_uid",
            "nunique",
        ),
        videos=(
            "video_uid",
            "nunique",
        ),
        minimum_turns=(
            "n_turns",
            "min",
        ),
        median_turns=(
            "n_turns",
            "median",
        ),
    )
)

organization_coverage

,rows,conversations,videos,minimum_turns,median_turns
trajectory_type,,,,,
expert,51,51,15,10,15.0
joint,79,79,16,11,24.0
partner,48,48,15,10,15.0


In [18]:
turns_dynamic["stage"] = np.select(
    [
        turns_dynamic[
            "_conversation_time"
        ] < 1 / 3,
        turns_dynamic[
            "_conversation_time"
        ] < 2 / 3,
    ],
    [
        "Beginning",
        "Middle",
    ],
    default="End",
)

In [19]:
stage_group_columns = [
    "video_uid",
    "conversation_uid",
    "conversation_id",
    "level_label",
    "stage",
]

organization_stage, lag_curves_stage = (
    calculate_organization_table(
        turn_table=turns_dynamic,
        embeddings=X_turns,
        group_columns=(
            stage_group_columns
        ),
        n_shuffles=500,
        maximum_lag=3,
        minimum_turns=8,
        random_seed=43,
    )
)

In [20]:
organization_stage.to_csv(
    EXPORT_DIR
    / "organization_stage.csv",
    index=False,
)

lag_curves_stage.to_csv(
    EXPORT_DIR
    / "organization_stage_lag_curves.csv",
    index=False,
)

who conversation multilevel models

In [21]:
gap_mapping = {
    "expert": 0,
    "graduate": 1,
    "graduate student": 1,
    "undergraduate": 2,
    "college": 2,
    "college student": 2,
    "teenager": 3,
    "teen": 3,
    "child": 4,
}

organization_conversation[
    "level_key"
] = (
    organization_conversation[
        "level_label"
    ]
    .astype(str)
    .str.lower()
    .str.strip()
)

organization_conversation[
    "gap_score"
] = (
    organization_conversation[
        "level_key"
    ]
    .map(gap_mapping)
)

organization_conversation[
    "gap_c"
] = (
    organization_conversation[
        "gap_score"
    ]
    - organization_conversation[
        "gap_score"
    ].mean()
)

In [22]:
#fit one model per trajectory type and metric
import statsmodels.formula.api as smf


def fit_organization_model(
    data,
    metric,
    trajectory_type="joint",
):
    outcome = f"{metric}_z"

    model_data = data[
        data[
            "trajectory_type"
        ] == trajectory_type
    ].dropna(
        subset=[
            outcome,
            "gap_c",
            "video_uid",
        ]
    ).copy()

    model = smf.mixedlm(
        formula=(
            f"{outcome} ~ gap_c"
        ),
        data=model_data,
        groups=model_data[
            "video_uid"
        ],
        re_formula="1",
    )

    result = model.fit(
        reml=False,
        method="lbfgs",
        maxiter=2000,
    )

    return result, model_data

In [23]:
locality_model, locality_data = (
    fit_organization_model(
        organization_conversation,
        metric="temporal_locality",
        trajectory_type="joint",
    )
)

print(
    locality_model.summary()
)

              Mixed Linear Model Regression Results
Model:            MixedLM Dependent Variable: temporal_locality_z
No. Observations: 79      Method:             ML                 
No. Groups:       16      Scale:              2.4568             
Min. group size:  4       Log-Likelihood:     -154.6603          
Max. group size:  5       Converged:          Yes                
Mean group size:  4.9                                            
-------------------------------------------------------------------
                Coef.   Std.Err.     z      P>|z|   [0.025   0.975]
-------------------------------------------------------------------
Intercept       3.669      0.274   13.370   0.000    3.131    4.207
gap_c           0.062      0.124    0.501   0.616   -0.181    0.306
Group Var       0.705      0.302                                   



/Users/yume/PycharmProjects/horizons_2/.venv/lib/python3.12/site-packages/statsmodels/regression/mixed_linear_model.py:1634: UserWarning: Random effects covariance is singular
  warnings.warn(msg)


## visualizations

In [24]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

apply_style()

level_order = [
    "child",
    "teenager",
    "undergraduate",
    "graduate",
    "expert",
]

stage_order = [
    "Beginning",
    "Middle",
    "End",
]

palette = expertise_palette(level_order)

gap_mapping = {
    "expert": 0,
    "graduate": 1,
    "graduate student": 1,
    "undergraduate": 2,
    "college": 2,
    "college student": 2,
    "teenager": 3,
    "teen": 3,
    "child": 4,
}


def add_plot_columns(data):
    data = data.copy()

    data["level_key"] = (
        data["level_label"]
        .astype(str)
        .str.lower()
        .str.strip()
        .replace({
            "teen": "teenager",
            "college": "undergraduate",
            "college student": "undergraduate",
            "graduate student": "graduate",
        })
    )

    data["gap_score"] = (
        data["level_key"]
        .map(gap_mapping)
    )

    return data

In [25]:
organization_conversation = (
    add_plot_columns(
        organization_conversation
    )
)

lag_curves_conversation = (
    add_plot_columns(
        lag_curves_conversation
    )
)

organization_stage = (
    add_plot_columns(
        organization_stage
    )
)

In [26]:
organization_metrics = {
    "temporal_locality_z": (
        "Temporal locality"
    ),
    "lag_growth_slope_z": (
        "Lag-growth structure"
    ),
    "lag_contrast_z": (
        "Short–long lag contrast"
    ),
    "recurrence_determinism_z": (
        "Recurrence determinism"
    ),
}

joint_organization = (
    organization_conversation[
        organization_conversation[
            "trajectory_type"
        ] == "joint"
    ]
    .copy()
)

In [27]:
fig, axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
    sharex=True,
)

axes = axes.flatten()

for ax, (metric, label) in zip(
    axes,
    organization_metrics.items(),
):
    metric_data = joint_organization[
        [
            "level_key",
            metric,
        ]
    ].dropna()

    sns.boxplot(
        data=metric_data,
        x="level_key",
        y=metric,
        order=level_order,
        hue="level_key",
        hue_order=level_order,
        palette=palette,
        showfliers=False,
        legend=False,
        ax=ax,
    )

    sns.stripplot(
        data=metric_data,
        x="level_key",
        y=metric,
        order=level_order,
        color="black",
        alpha=0.45,
        size=3,
        jitter=0.18,
        ax=ax,
    )

    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
    )

    ax.set_title(label)
    ax.set_xlabel("")
    ax.set_ylabel(
        "Organization relative to shuffle (z)"
    )

    ax.tick_params(
        axis="x",
        rotation=25,
    )

fig.suptitle(
    "Temporal organization of joint semantic trajectories",
    y=1.02,
)

plt.tight_layout()
plt.show()

In [28]:
trajectory_metrics = (
    organization_conversation[
        [
            "trajectory_type",
            "level_key",
            "temporal_locality_z",
            "lag_growth_slope_z",
        ]
    ]
    .dropna()
    .melt(
        id_vars=[
            "trajectory_type",
            "level_key",
        ],
        value_vars=[
            "temporal_locality_z",
            "lag_growth_slope_z",
        ],
        var_name="metric",
        value_name="organization_z",
    )
)

trajectory_metrics["metric"] = (
    trajectory_metrics["metric"]
    .replace({
        "temporal_locality_z": (
            "Temporal locality"
        ),
        "lag_growth_slope_z": (
            "Lag-growth structure"
        ),
    })
)

In [29]:
g = sns.catplot(
    data=trajectory_metrics,
    x="level_key",
    y="organization_z",
    hue="level_key",
    hue_order=level_order,
    palette=palette,
    order=level_order,
    row="trajectory_type",
    col="metric",
    kind="box",
    showfliers=False,
    height=3.2,
    aspect=1.25,
    legend=False,
)

for ax in g.axes.flat:
    ax.axhline(
        0,
        color="gray",
        linestyle="--",
        linewidth=1,
    )

    ax.tick_params(
        axis="x",
        rotation=25,
    )

g.set_axis_labels(
    "",
    "Organization relative to shuffle (z)",
)

g.set_titles(
    row_template="{row_name}",
    col_template="{col_name}",
)

plt.show()

In [30]:
[
    "video_uid",
    "conversation_uid",
    "conversation_id",
    "level_label",
    "trajectory_type",
    "method",
    "n_turns",
    "score_observed",
    "score_shuffle_mean",
    "score_shuffle_sd",
    "organization_gain",
    "organization_z",
]

['video_uid',
 'conversation_uid',
 'conversation_id',
 'level_label',
 'trajectory_type',
 'method',
 'n_turns',
 'score_observed',
 'score_shuffle_mean',
 'score_shuffle_sd',
 'organization_gain',
 'organization_z']

In [31]:
from sklearn.model_selection import GroupKFold

outer_cv = GroupKFold(
    n_splits=5
)